# 069: KmerSeek HP — Unique Hits & SCOPe Class Enrichment (excl. gray zone, best FAM k)

KmerSeek HP unique hits analysis at the **best family-level k** (k=29) under the
**exclude-gray-zone** convention (FoldSeek/TEA paper standard).

The best FAM k (k=29, n=164 covered queries, FAM AUC≈0.991) is taken from the
Bonferroni-corrected sweep in notebook 068.

**Gray-zone exclusion**: cross-fold hits that are NOT cross-superfamily are
excluded from false positives (paper convention). This is stricter than the
legacy `all_fps` convention (notebook 067).

## 1. Imports & configuration

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, gc
from pathlib import Path
from collections import Counter

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.append(str(Path.cwd()))
from scope_kmerseek_utils import (
    EVAL_USECOLS,
    BENCH_DIR, TEA_DIR,
    SCOPE_CLASS_DESCRIPTIONS,
    load_eval, load_baselines, add_composite_scores_scope,
    recompute_bonferroni,
    eval_tsv_to_rocx, rocx_restrict, sensitivity_stats,
    plot_sensitivity_from_rocx, compute_auc,
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.dpi'] = 120

K_UNIQUE        = [29]          # best FAM k from excl_gz Bonferroni sweep (notebook 068)
EXCLUDE_GRAY_ZONE = True
BONF_THRESHOLD  = 0.05
SCORE_COL       = 'jaccard'

print('SCOPe class descriptions:')
for cls, desc in SCOPE_CLASS_DESCRIPTIONS.items():
    print(f'  {cls}: {desc}')

## 2. Load baselines and KmerSeek k=29

In [ ]:
foldseek, tea_all = load_baselines(TEA_DIR)
ref_queries = set(foldseek['NAME'].to_list())

print(f'FoldSeek queries: {len(foldseek):,}')
print(f'TEA queries:      {len(tea_all):,}')
print(f'Reference query set: {len(ref_queries):,}')

for name, df in [('FoldSeek', foldseek), ('TEA', tea_all)]:
    _, _, fam  = sensitivity_stats(df, 'FAM')
    _, _, sfam = sensitivity_stats(df, 'SFAM')
    print(f'{name}: FAM AUC={fam:.4f}  SFAM AUC={sfam:.4f}')

In [ ]:
km_rocx = {}
km_rocx_r = {}

for k in K_UNIQUE:
    print(f'Loading k={k}...')
    df = load_eval(k, columns=EVAL_USECOLS)
    df = add_composite_scores_scope(df)
    df = recompute_bonferroni(df)
    print(f'  {len(df):,} pairs before filter')

    df = df.filter(pl.col('bonferroni_correct') < BONF_THRESHOLD)
    print(f'  {len(df):,} pairs after Bonferroni<{BONF_THRESHOLD} filter')

    rocx = eval_tsv_to_rocx(df, score_col=SCORE_COL, ascending=False,
                             exclude_gray_zone=EXCLUDE_GRAY_ZONE)
    km_rocx[k]   = rocx
    km_rocx_r[k] = rocx.filter(pl.col('NAME').is_in(ref_queries))
    del df; gc.collect()

    _, _, fam  = sensitivity_stats(km_rocx_r[k], 'FAM')
    _, _, sfam = sensitivity_stats(km_rocx_r[k], 'SFAM')
    print(f'  FAM AUC (Jaccard, excl_gz): {fam:.4f}')
    print(f'  SFAM AUC (Jaccard, excl_gz): {sfam:.4f}')

In [ ]:
# AUC comparison table: FAM, SFAM, FOLD
print(f'{"Method":<35} {"FAM AUC":>9} {"SFAM AUC":>9} {"FOLD AUC":>9}  n_queries')
print('-' * 75)

for name, df in [('FoldSeek', foldseek), ('TEA', tea_all)]:
    _, _, fam  = sensitivity_stats(df, 'FAM')
    _, _, sfam = sensitivity_stats(df, 'SFAM')
    _, _, fold = sensitivity_stats(df, 'FOLD')
    n = df['NAME'].n_unique()
    print(f'{name:<35} {fam:>9.4f} {sfam:>9.4f} {fold:>9.4f}  {n:,}')

print()
for k in K_UNIQUE:
    rocx_cov = km_rocx_r[k]
    _, _, fam  = sensitivity_stats(rocx_cov, 'FAM')
    _, _, sfam = sensitivity_stats(rocx_cov, 'SFAM')
    _, _, fold = sensitivity_stats(rocx_cov, 'FOLD')
    n = rocx_cov['NAME'].n_unique()
    label = f'KmerSeek k={k} Bonf<{BONF_THRESHOLD} excl_gz'
    print(f'{label:<35} {fam:>9.4f} {sfam:>9.4f} {fold:>9.4f}  {n:,}')

## 3. Sensitivity curves: k=29 vs FoldSeek vs TEA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_k = {29: 'tomato'}

for ax, (lc, ll) in zip(axes, [('FAM', 'Family'), ('SFAM', 'Superfamily')]):
    frac, sens, auc = sensitivity_stats(foldseek, lc)
    ax.plot(frac, sens, 'k-', lw=2.5, label=f'FoldSeek (AUC={auc:.3f})')

    frac, sens, auc = sensitivity_stats(tea_all, lc)
    ax.plot(frac, sens, 'k--', lw=2.5, label=f'TEA (AUC={auc:.3f})')

    for k in K_UNIQUE:
        frac, sens, auc = sensitivity_stats(km_rocx_r[k], lc)
        ax.plot(frac, sens, color=colors_k[k], lw=2.5,
                label=f'KmerSeek k={k} Bonf<{BONF_THRESHOLD} (AUC={auc:.3f})')

    ax.set_xlabel('Fraction of Queries', fontsize=12)
    ax.set_ylabel(f'Sensitivity to First FP ({ll})', fontsize=12)
    ax.set_title(f'{ll}-level Sensitivity Curves', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle(
    f'KmerSeek HP k=29 (Bonferroni<{BONF_THRESHOLD}, excl. gray zone) vs FoldSeek & TEA — SCOPe40',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/069_sensitivity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Unique hits analysis

A query is "detected" if its sensitivity at the family level > 0 (≥1 true homolog found
before first FP). Unique = detected by KmerSeek but NOT by FoldSeek or TEA.

In [ ]:
def detected_queries(rocx_df: pl.DataFrame, level_col: str = 'FAM') -> set:
    return set(rocx_df.filter(pl.col(level_col) > 0)['NAME'].to_list())


fs_detected   = detected_queries(foldseek)
tea_detected  = detected_queries(tea_all)
base_detected = fs_detected | tea_detected

print(f'FoldSeek detected:   {len(fs_detected):,} queries')
print(f'TEA detected:        {len(tea_detected):,} queries')
print(f'Either baseline:     {len(base_detected):,} queries')
print()

for k in K_UNIQUE:
    km_det  = detected_queries(km_rocx[k])
    unique  = km_det - base_detected
    pct_u   = 100 * len(unique) / len(km_det) if km_det else 0
    overlap = km_det & base_detected
    print(f'k={k} (Bonferroni<{BONF_THRESHOLD}, excl_gz):')
    print(f'  KmerSeek detected:          {len(km_det):,} queries')
    print(f'  Unique to KmerSeek:         {len(unique):,} ({pct_u:.1f}% of KmerSeek hits)')
    print(f'  Shared with FoldSeek/TEA:   {len(overlap):,}')

## 5. SCOPe class distribution of unique vs all hits

In [ ]:
def scop_class_from_rocx(rocx_df: pl.DataFrame) -> list:
    return (
        rocx_df
        .filter(pl.col('SCOP').str.len_chars() > 0)
        ['SCOP']
        .str.slice(0, 1)
        .to_list()
    )

k_plot = K_UNIQUE[0]
km_det_all    = detected_queries(km_rocx[k_plot])
km_det_unique = km_det_all - base_detected

rocx_all    = km_rocx[k_plot]
rocx_unique = rocx_all.filter(pl.col('NAME').is_in(km_det_unique))

classes_km_all    = scop_class_from_rocx(rocx_all.filter(pl.col('NAME').is_in(km_det_all)))
classes_km_unique = scop_class_from_rocx(rocx_unique)
classes_fs        = scop_class_from_rocx(foldseek.filter(pl.col('FAM') > 0))
classes_tea       = scop_class_from_rocx(tea_all.filter(pl.col('FAM') > 0))

def class_pct(classes: list) -> dict:
    total = len(classes)
    if total == 0:
        return {}
    c = Counter(classes)
    return {k: v / total * 100 for k, v in c.items()}

print(f'KmerSeek k={k_plot} unique: {len(km_det_unique)} queries')
print(f'KmerSeek k={k_plot} all:    {len(km_det_all)} queries')
print(f'FoldSeek detected:          {len(classes_fs)} queries')

In [ ]:
from scipy.stats import fisher_exact

count_unique = Counter(classes_km_unique)
n_hp = len(classes_km_unique)
n_fs = len(classes_fs)

def enrichment_pval(cls):
    n_hp_cls = sum(1 for c in classes_km_unique if c == cls)
    n_fs_cls = sum(1 for c in classes_fs        if c == cls)
    _, pval = fisher_exact(
        [[n_hp_cls, n_hp - n_hp_cls],
         [n_fs_cls, n_fs - n_fs_cls]],
        alternative='greater')
    return pval

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
classes_sorted = sorted(count_unique, key=lambda c: -count_unique[c])
counts_sorted  = [count_unique[c] for c in classes_sorted]
bars = ax.bar(classes_sorted, counts_sorted, color='teal', alpha=0.8)
for bar, val in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xlabel('SCOPe Class', fontsize=12)
ax.set_ylabel('Number of Unique Detections', fontsize=12)
ax.set_title(f'KmerSeek-unique hits by class\n(k={k_plot}, Bonf<{BONF_THRESHOLD}, excl_gz)',
             fontsize=12, fontweight='bold')
ax.set_xticklabels([f'{c}\n{SCOPE_CLASS_DESCRIPTIONS.get(c,c)[:20]}' for c in classes_sorted], fontsize=8)

ax = axes[1]
all_cls_set = sorted(set(classes_sorted) | set(Counter(classes_fs).keys()))
pct_unique = class_pct(classes_km_unique)
pct_fs     = class_pct(classes_fs)
x = np.arange(len(all_cls_set)); w = 0.35
ax.bar(x - w/2, [pct_unique.get(c, 0) for c in all_cls_set], w,
       color='teal', alpha=0.8, label=f'KmerSeek-unique (n={n_hp})')
ax.bar(x + w/2, [pct_fs.get(c, 0)     for c in all_cls_set], w,
       color='steelblue', alpha=0.6, label=f'FoldSeek detected (n={n_fs})')
ax.set_xticks(x)
ax.set_xticklabels([f'{c}\n{SCOPE_CLASS_DESCRIPTIONS.get(c,c)[:20]}' for c in all_cls_set], fontsize=8)
ax.set_ylabel('% of detected queries', fontsize=12)
ax.set_title('SCOPe class % — KmerSeek-unique vs FoldSeek', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../figures/069_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fisher's exact test: enrichment by SCOPe class
km_unique_ids = km_det_all - base_detected
km_shared_ids = km_det_all & base_detected

def cls_from_rocx(rocx_df, query_set):
    sub = (rocx_df
           .filter(pl.col('NAME').is_in(query_set))
           .select(['NAME', 'SCOP'])
           .unique()
           .with_columns(pl.col('SCOP').str.slice(0, 1).alias('cls')))
    return dict(zip(sub['NAME'].to_list(), sub['cls'].to_list()))

cls_unique = cls_from_rocx(km_rocx[k_plot], km_unique_ids)
cls_shared = cls_from_rocx(km_rocx[k_plot], km_shared_ids)
all_cls    = sorted(set(cls_unique.values()) | set(cls_shared.values()))

print(f'KmerSeek-unique: {len(km_unique_ids)} queries')
print(f'KmerSeek-shared: {len(km_shared_ids)} queries')
print()
print(f'{"Class":<6} {"Description":<35} {"Unique":>7} {"Shared":>7} {"p-enrich":>10}')
print('-' * 70)
for cls in sorted(all_cls):
    n_u = sum(1 for v in cls_unique.values() if v == cls)
    n_s = sum(1 for v in cls_shared.values() if v == cls)
    pval = enrichment_pval(cls)
    desc = SCOPE_CLASS_DESCRIPTIONS.get(cls, cls)[:32]
    sig  = ' *' if pval < 0.05 else ''
    print(f'{cls:<6} {desc:<35} {n_u:>7} {n_s:>7} {pval:>10.4f}{sig}')

## 6. Families unique to KmerSeek

In [ ]:
def get_families(rocx_df: pl.DataFrame, query_set: set) -> list:
    subset = rocx_df.filter(pl.col('NAME').is_in(query_set))
    families = []
    for scop in subset['SCOP'].to_list():
        if not scop:
            continue
        parts = scop.split('.')
        if len(parts) >= 4:
            families.append('.'.join(parts[:4]))
        elif len(parts) == 3:
            families.append(scop)
    return families

km_unique_ids = km_det_all - base_detected
fams_unique   = get_families(km_rocx[k_plot], km_unique_ids)
fams_all_km   = get_families(km_rocx[k_plot], km_det_all)

unique_families = set(fams_unique)
print(f'Families in KmerSeek-unique detections: {len(unique_families)}')
print(f'Total KmerSeek families (detected):     {len(set(fams_all_km))}')
pct = 100 * len(unique_families) / len(set(fams_all_km)) if fams_all_km else 0
print(f'Fraction unique families: {pct:.1f}%')
fam_counts = Counter(fams_unique)

In [ ]:
class_fam_counts = {}
for fam in unique_families:
    cls = fam[0] if fam else '?'
    class_fam_counts[cls] = class_fam_counts.get(cls, 0) + 1

classes_sorted = sorted(class_fam_counts, key=lambda c: -class_fam_counts[c])
counts_sorted  = [class_fam_counts[c] for c in classes_sorted]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(classes_sorted, counts_sorted, color='teal', alpha=0.85, edgecolor='white')
for bar, val in zip(bars, counts_sorted):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_xlabel('SCOPe Class', fontsize=12)
ax.set_ylabel('Number of Unique Families', fontsize=12)
ax.set_title(
    f'HP-Unique SCOP Families by Class (k={k_plot}, Bonf<{BONF_THRESHOLD}, excl_gz)\n'
    f'n={len(unique_families)} unique families',
    fontsize=12, fontweight='bold')
ax.set_xticklabels(
    [f'{c}\n{SCOPE_CLASS_DESCRIPTIONS.get(c,c)[:25]}' for c in classes_sorted], fontsize=9)
plt.tight_layout()
plt.savefig('../figures/069_unique_families_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. SCOP coverage: unique vs all hits

In [ ]:
def _scop_ids_at_level(query_set, n_parts):
    ids = set()
    for scop in km_rocx[k_plot].filter(pl.col('NAME').is_in(query_set))['SCOP'].to_list():
        if scop:
            parts = scop.split('.')
            if len(parts) >= n_parts:
                ids.add('.'.join(parts[:n_parts]))
    return ids

levels_def  = list(reversed([('Class', 1), ('Fold', 2), ('Superfamily', 3), ('Family', 4)]))
level_names = [l for l, _ in levels_def]
n_unique_ids = [len(_scop_ids_at_level(km_unique_ids, n)) for _, n in levels_def]
n_all_ids    = [len(_scop_ids_at_level(km_det_all,    n)) for _, n in levels_def]
n_shared_ids = [a - u for a, u in zip(n_all_ids, n_unique_ids)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x = np.arange(len(level_names)); w = 0.5
ax.bar(x, n_unique_ids, w, label='KmerSeek-unique', color='#d73027', alpha=0.85)
ax.bar(x, n_shared_ids, w, bottom=n_unique_ids, label='Shared with baselines', color='#4575b4', alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(level_names, fontsize=11)
ax.set_ylabel('Number of unique SCOPe IDs', fontsize=11)
ax.set_title('SCOP coverage: unique vs shared', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)

ax = axes[1]
pct_unique_ids = [u / a * 100 if a > 0 else 0 for u, a in zip(n_unique_ids, n_all_ids)]
ax.bar(x, pct_unique_ids, w, color='#d73027', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(level_names, fontsize=11)
ax.set_ylabel('% of KmerSeek hits that are unique', fontsize=11)
ax.set_title('% Unique SCOP IDs per level', fontsize=12, fontweight='bold')
for i, (pct, n) in enumerate(zip(pct_unique_ids, n_unique_ids)):
    ax.text(i, pct + 0.5, f'{n}\n({pct:.0f}%)', ha='center', fontsize=9)

plt.suptitle(f'SCOP Coverage — KmerSeek k={k_plot} (excl. gray zone)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/069_scop_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Top unique families

In [ ]:
import pandas as pd

TOP_N = 20
top_fams = fam_counts.most_common(TOP_N)

CLASS_COLORS = {
    'a': '#e41a1c', 'b': '#377eb8', 'c': '#4daf4a',
    'd': '#984ea3', 'e': '#ff7f00', 'f': '#a65628', 'g': '#f781bf',
}

fig_fam, ax_fam = plt.subplots(figsize=(8, 7))
fam_labels  = [fam for fam, _ in top_fams]
fam_vals    = [n   for _,   n in top_fams]
bar_colors  = [CLASS_COLORS.get(fam[0], 'grey') for fam in fam_labels]

bars = ax_fam.barh(fam_labels[::-1], fam_vals[::-1], color=bar_colors[::-1], alpha=0.85)
for bar, val in zip(bars, fam_vals[::-1]):
    ax_fam.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
                f'n={val}', va='center', fontsize=9, fontweight='bold')

handles = [mpatches.Patch(color=v, label=f'Class {k}: {SCOPE_CLASS_DESCRIPTIONS.get(k,k)[:30]}')
           for k, v in CLASS_COLORS.items() if k in class_fam_counts]
ax_fam.legend(handles=handles, fontsize=8, loc='lower right')
ax_fam.set_xlabel('Number of unique hit queries', fontsize=11)
ax_fam.set_title(f'Top {TOP_N} Unique SCOP Families (KmerSeek k={k_plot}, excl_gz)',
                 fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/069_top_unique_families.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print('=' * 90)
print(f'SUMMARY: KmerSeek HP k={k_plot} — excl. gray zone, Bonferroni<{BONF_THRESHOLD}')
print('=' * 90)

print('\n### Sensitivity AUC (covered-query AUC)\n')
print(f'  {"Method":<45} {"FAM":>8} {"SFAM":>8} {"FOLD":>8}  n_queries')
print(f'  {"-"*80}')
for name, df in [('FoldSeek', foldseek), ('TEA', tea_all)]:
    _, _, fa = sensitivity_stats(df, 'FAM')
    _, _, sf = sensitivity_stats(df, 'SFAM')
    _, _, fo = sensitivity_stats(df, 'FOLD')
    print(f'  {name:<45} {fa:>8.4f} {sf:>8.4f} {fo:>8.4f}  {df["NAME"].n_unique():,}')
_, _, fa = sensitivity_stats(km_rocx_r[k_plot], 'FAM')
_, _, sf = sensitivity_stats(km_rocx_r[k_plot], 'SFAM')
_, _, fo = sensitivity_stats(km_rocx_r[k_plot], 'FOLD')
label = f'KmerSeek k={k_plot} Bonf<{BONF_THRESHOLD} excl_gz'
print(f'  {label:<45} {fa:>8.4f} {sf:>8.4f} {fo:>8.4f}  {km_rocx_r[k_plot]["NAME"].n_unique():,}')

print('\n### Unique hits summary\n')
print(f'  KmerSeek-unique queries (FAM level): {len(km_unique_ids)}')
print(f'  Unique SCOP families:                {len(unique_families)}')
top3_cls = sorted(class_fam_counts, key=lambda c: -class_fam_counts[c])[:3]
print(f'  Top enriched classes: {top3_cls}')